## Zadanie 3: 

W 2014 roku architektura VGG zrewolucjonizowała sieci CNN. Jej twórcy zrezygnowali z wymyślania skomplikowanych i różnych od siebie warstw. Zamiast tego postawili na **prostotę i głębokość**. 

Zauważyli, że lepiej użyć dwóch warstw splotowych z małymi filtrami (3x3) jedna po drugiej, niż jednej warstwy z dużym filtrem (5x5). Aby kod był czytelny, VGG buduje się z powtarzalnych "bloków".

Standardowy blok VGG składa się z:
1. Warstwy splotowej (filtr 3x3, padding 1 - aby zachować rozmiar obrazu)
2. Aktywacji ReLU
3. Drugiej warstwy splotowej (filtr 3x3, padding 1)
4. Aktywacji ReLU
5. Warstwy MaxPool (redukującej wymiar o połowę)

**Twoje zadanie:**
1. Uzupełnij funkcję `create_vgg_block`, wykorzystując `nn.Sequential`. Pamiętaj, aby druga warstwa splotowa w bloku przyjmowała tyle kanałów, ile zwróciła pierwsza!
2. Przeanalizuj klasę `MiniVGG`, która z tych bloków korzysta.
3. Skompiluj model i sprawdź, czy kształt wyjściowy się zgadza.

In [10]:
import torch
import torch.nn as nn

In [12]:
def create_vgg_block(in_channels, out_channels):
    """
    Funkcja generująca jeden blok architektury VGG.
    Wykorzystujemy nn.Sequential do zgrupowania warstw.
    """
    block = nn.Sequential(
        # TODO 1: Pierwszy splot (kernel_size=3, padding=1)
        nn.Conv2d(in_channels=..., out_channels=..., kernel_size=..., padding=...),
        nn.ReLU(),
        
        # TODO 2: Drugi splot (uwaga na liczbę kanałów wejściowych dla tej warstwy!)
        nn.Conv2d(in_channels=..., out_channels=..., kernel_size=..., padding=...),
        nn.ReLU(),
        
        # TODO 3: Pooling zmniejszający wymiar przestrzenny 2 razy
        nn.MaxPool2d(kernel_size=2, stride=2)
    )
    return block

In [14]:
class MiniVGG(nn.Module):
    def __init__(self):
        super(MiniVGG, self).__init__()
        
        # Wejście: np. CIFAR-10 (3 kanały, 32x32 piksele)
        # Budujemy sieć z naszych klocków!
        self.block1 = create_vgg_block(in_channels=3, out_channels=16) 
        # Rozmiar po block1: 16 kanałów, 16x16 pikseli (bo jeden MaxPool)
        
        self.block2 = create_vgg_block(in_channels=16, out_channels=32)
        # Rozmiar po block2: 32 kanały, 8x8 pikseli (bo kolejny MaxPool)
        
        # Klasyfikator również możemy ubrać w nn.Sequential
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.5), # Dodajemy Dropout zapobiegawczo
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.classifier(x)
        return x

In [18]:
# Testujemy naszego Mini-VGG
dummy_input = torch.randn(1, 3, 32, 32)
model_vgg = MiniVGG()

print("Przetwarzanie przez MiniVGG...")
# Wydrukujmy też sam model, aby zobaczyć jak pięknie i czytelnie wygląda dzięki nn.Sequential
print(model_vgg) 

output = model_vgg(dummy_input)
print("\nKształt wyjścia:", output.shape)

TypeError: unsupported operand type(s) for %: 'ellipsis' and 'int'